# 10 · Validação manual do CryptoBERT

O CryptoBERT não foi treinado nos nossos textos. Aqui uma amostra deles é rotulada à mão, sem ver a nota do modelo, e as duas classificações são comparadas.
- Amostra de 102 textos: 17 em cada combinação de fonte (notícia, tweet) e classe do CryptoBERT (bearish, neutro, bullish), com as notas da etapa 04.
- Classe do CryptoBERT: bearish se a nota ≤ −1/3, bullish se ≥ 1/3, neutro entre os dois (mesmo corte da etapa 04).
- Concordância: acurácia, kappa de Cohen e kappa ponderado (que trata bearish × bullish como erro maior que bearish × neutro).

Não precisa de GPU. Roda no computador ou no Kaggle (com a pasta do projeto como Dataset, incluindo `results/04_sentimento/nota_por_texto.csv` e a planilha preenchida em `data/validacao/rotulos.xlsx`).

In [ ]:
import os, sys, glob
achados = glob.glob('../src/tcc/config.py') + glob.glob('/kaggle/input/**/src/tcc/config.py', recursive=True)
RAIZ = achados[0].rsplit('/src/', 1)[0]              # pasta do projeto (no PC ou no dataset do Kaggle)
sys.path.insert(0, os.path.join(RAIZ, 'src'))
import warnings; warnings.filterwarnings('ignore')
import json, shutil
from pathlib import Path
import pandas as pd, matplotlib.pyplot as plt
from tcc import config, validacao

In [ ]:
POR_GRUPO = 17
SEMENTE   = 42

def achar(*partes):
    # procura no projeto e, no Kaggle, em /kaggle/working e nos datasets
    for base in [Path(RAIZ), config.GRAVAVEL, *Path('/kaggle/input').glob('**/')]:
        if (base / Path(*partes)).exists():
            return base / Path(*partes)
    return config.GRAVAVEL / Path(*partes)

NOTAS    = achar('results', '04_sentimento', 'nota_por_texto.csv')
PLANILHA = achar('data', 'validacao', 'rotulos.xlsx')     # tem os textos: fica fora do Git
print('Notas:', NOTAS); print('Planilha:', PLANILHA)
SAIDA = config.PASTA_RESULTADOS / '10_validacao_manual'
SAIDA.mkdir(parents=True, exist_ok=True)

## 1. Amostra

In [ ]:
amostra = validacao.amostra(pd.read_csv(NOTAS), POR_GRUPO, SEMENTE)
if not PLANILHA.exists():
    validacao.planilha(amostra, PLANILHA)
    print('Planilha criada:', PLANILHA)
amostra.groupby(['fonte', 'classe_cryptobert']).size().unstack()

## 2. Concordância

Só roda com a coluna `minha_classe` toda preenchida.

In [ ]:
rotulos = pd.read_excel(PLANILHA)[['id', 'minha_classe']]
rotulos['minha_classe'] = rotulos['minha_classe'].fillna('').astype(str).str.strip().str.lower()
faltam = (~rotulos['minha_classe'].isin(validacao.CLASSES)).sum()
assert faltam == 0, f'{faltam} textos sem classe válida em {PLANILHA.name}'

tab = amostra.merge(rotulos, on='id')
resumo = pd.DataFrame([{'fonte': 'todas', **validacao.concordancia(tab.minha_classe, tab.classe_cryptobert)}] +
                      [{'fonte': f, **validacao.concordancia(g.minha_classe, g.classe_cryptobert)} for f, g in tab.groupby('fonte')])
resumo.round(3).to_csv(SAIDA / 'resumo.csv', index=False)
tab[['id', 'fonte', 'moeda', 'data', 'nota', 'classe_cryptobert', 'minha_classe']].to_csv(SAIDA / 'rotulos.csv', index=False)
resumo.round(3)

In [ ]:
m = validacao.matriz(tab.minha_classe, tab.classe_cryptobert)
m.to_csv(SAIDA / 'matriz_confusao.csv')

fig, ax = plt.subplots(figsize=(4.5, 4))
ax.imshow(m.values, cmap='Blues')
for i in range(3):
    for j in range(3):
        ax.text(j, i, m.values[i, j], ha='center', va='center', color='white' if m.values[i, j] > m.values.max() / 2 else 'black')
ax.set_xticks(range(3)); ax.set_xticklabels(m.columns); ax.set_yticks(range(3)); ax.set_yticklabels(m.index)
ax.set_xlabel('CryptoBERT'); ax.set_ylabel('rótulo manual')
plt.tight_layout(); plt.savefig(SAIDA / 'matriz_confusao.png', dpi=130); plt.show()
m

## 3. Onde discordam

Nota média do CryptoBERT para cada rótulo manual. Se o modelo acompanha o leitor, a nota sobe de bearish para bullish.

In [ ]:
tab.groupby('minha_classe')['nota'].agg(['count', 'mean', 'median']).reindex(validacao.CLASSES).round(3)

In [ ]:
(SAIDA / 'config.json').write_text(json.dumps({'por_grupo': POR_GRUPO, 'semente': SEMENTE, 'limite_classe': validacao.LIMITE,
    'classes': validacao.CLASSES, 'modelo': config.MODELO_SENTIMENTO}, indent=2))
print('Pronto:', shutil.make_archive(str(SAIDA), 'zip', SAIDA))